# Biology and the Scientific Understanding of Living Order Workflow

This notebook scaffold supports the article **Biology and the Scientific Understanding of Living Order**. It can be expanded with homeostasis, recovery indices, growth fitting, logistic constraint, feedback, biological networks, condition scoring, and provenance notes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
home = pd.read_csv(article_dir / 'data' / 'homeostasis_scenarios.csv')
rows = []
for _, s in home.iterrows():
    final_state = s['setpoint'] + (s['initial_value'] - s['setpoint']) * np.exp(-s['correction_rate'] * s['time_end'])
    recovery = 1 - abs(final_state - s['setpoint']) / abs(s['initial_value'] - s['setpoint'])
    rows.append({'scenario': s['scenario'], 'final_state': final_state, 'recovery_index': recovery})
pd.DataFrame(rows).round(5)

In [ ]:
growth = pd.read_csv(article_dir / 'data' / 'growth_observations.csv')
growth_rows = []
for condition, group in growth.groupby('condition'):
    slope, intercept = np.polyfit(group['time'], np.log(group['abundance']), 1)
    growth_rows.append({'condition': condition, 'growth_rate': slope, 'N0': np.exp(intercept), 'doubling_time': np.log(2) / slope})
pd.DataFrame(growth_rows).round(5)

In [ ]:
feedback = pd.read_csv(article_dir / 'data' / 'feedback_scenarios.csv')
feedback['deviation'] = feedback['state'] - feedback['setpoint']
feedback['corrective_response'] = feedback['feedback_gain'] * (feedback['setpoint'] - feedback['state'])
feedback.round(5)

In [ ]:
edges = pd.read_csv(article_dir / 'data' / 'network_edges.csv')
nodes = sorted(set(edges['source']).union(edges['target']))
centrality_rows = []
for node in nodes:
    mask = (edges['source'] == node) | (edges['target'] == node)
    centrality_rows.append({'node': node, 'degree': int(mask.sum()), 'weighted_degree': edges.loc[mask, 'interaction_weight'].sum()})
pd.DataFrame(centrality_rows).sort_values('weighted_degree', ascending=False).round(3)

In [ ]:
condition = pd.read_csv(article_dir / 'data' / 'living_order_condition_sites.csv')
condition['living_order_score'] = (
    0.17 * condition['homeostatic_regulation'] +
    0.16 * condition['metabolic_throughput'] +
    0.15 * condition['structural_integration'] +
    0.13 * condition['developmental_coordination'] +
    0.15 * condition['information_continuity'] +
    0.14 * condition['ecological_relation'] +
    0.10 * (1 - condition['stress_penalty'])
)
condition.sort_values('living_order_score', ascending=False).round(3)